# Phase 9–10 & 12: Temporal Fusion Transformer — Primary Model

This notebook covers the full TFT workflow:
1. **Phase 9** — Build `TimeSeriesDataSet`, train TFT, generate quantile predictions
2. **Phase 10** — Optuna hyperparameter search (30 trials), retrain best config, update `tft_config.yaml`
3. **Phase 12** — Interpretability: VSN variable importance, attention heatmaps, prediction interval calibration

**Why TFT?** It combines LSTM local processing, interpretable multi-head attention for long-range dependencies, variable selection networks (VSN), and quantile outputs for uncertainty estimation — ideal for M5's heterogeneous retail time series.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import optuna
from optuna.pruners import MedianPruner
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42)

PREDS_DIR  = Path('../outputs/predictions')
MODELS_DIR = Path('../outputs/models')
FIGS_DIR   = Path('../outputs/figures')
for d in [PREDS_DIR, MODELS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

Seed set to 42


Libraries loaded.


## 9.1 Load Feature Data

In [ ]:
# ── Memory-efficient data loading ─────────────────────────────────────────
# train.parquet has 56 M rows x 67 cols (~34 GB uncompressed).
# We load only required columns and optionally sample series.
# Set N_SERIES=None to use all 30,490 series (~8 GB for these cols).
# Aligns with 06_evaluation via outputs/predictions/tft_train_sample_ids.txt
N_SERIES  = 5000   # set to None for full dataset
LOAD_COLS = ['id', 'date', 'sales', 'cat_id', 'dept_id', 'store_id', 'state_id', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_month_start', 'is_month_end', 'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28', 'time_idx']

def _load_split(path, cols=LOAD_COLS, sample_ids=None):
    df = pd.read_parquet(path, columns=cols)
    df['id'] = df['id'].astype('category')
    if sample_ids is not None:
        df = df[df['id'].isin(sample_ids)].copy()
        df['id'] = df['id'].cat.remove_unused_categories()
    return df

# Stratified sample balanced across cat_id x store_id
_meta = pd.read_parquet('../data/features/train.parquet',
                        columns=['id','cat_id','store_id']).drop_duplicates('id')
if N_SERIES and N_SERIES < len(_meta):
    _meta = _meta.groupby(['cat_id','store_id'], group_keys=False).apply(
        lambda g: g.sample(max(1, round(N_SERIES * len(g) / len(_meta))),
                           random_state=42)
    ).head(N_SERIES)
    SAMPLE_IDS = set(_meta['id'].astype(str))
else:
    SAMPLE_IDS = None

train_df = _load_split('../data/features/train.parquet', sample_ids=SAMPLE_IDS)
val_df   = _load_split('../data/features/val.parquet',   sample_ids=SAMPLE_IDS)
test_df  = _load_split('../data/features/test.parquet',  sample_ids=SAMPLE_IDS)

print(f'N_SERIES={N_SERIES}  (None = all 30490)')
print(f'Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}')
print(f'RAM (train): {train_df.memory_usage(deep=True).sum()/1e6:.0f} MB')
if SAMPLE_IDS is not None:
    _sid_path = PREDS_DIR / 'tft_train_sample_ids.txt'
    _sid_path.write_text(chr(10).join(sorted(SAMPLE_IDS)) + chr(10))
    print(f'Saved {len(SAMPLE_IDS)} training id(s) to {_sid_path} (for 06_evaluation).')


N_SERIES=5000  (None = all 30490)
Train: (9285000, 32)  Val: (140000, 32)  Test: (140000, 32)
RAM (train): 910 MB
Saved 5000 training id(s) to ../outputs/predictions/tft_train_sample_ids.txt (for 06_evaluation).


In [3]:
# pytorch-forecasting requires string categoricals
# Convert integer-encoded categoricals back to strings so the library
# can create proper embeddings
for col in ['cat_id', 'dept_id', 'store_id', 'state_id']:
    if col in train_df.columns and pd.api.types.is_integer_dtype(train_df[col]):
        for df in [train_df, val_df, test_df]:
            df[col] = df[col].astype(str)

# Ensure time_idx is integer
for df in [train_df, val_df, test_df]:
    df['time_idx'] = df['time_idx'].astype(int)

print('Data types verified.')
print(train_df[['cat_id', 'dept_id', 'store_id', 'state_id']].dtypes)

Data types verified.
cat_id      object
dept_id     object
store_id    object
state_id    object
dtype: object


## 9.2 Build TimeSeriesDataSet

The TFT requires a `TimeSeriesDataSet` that encodes:
- **Static categoricals**: store/dept/category/state (never change per series)
- **Known future reals**: calendar, events, price (available for all future dates)
- **Unknown reals**: lag and rolling features (only available up to present)

In [4]:
from src.models.tft_model import (
    TIME_VARYING_KNOWN_REALS,
    TIME_VARYING_UNKNOWN_REALS,
    STATIC_CATEGORICALS,
    build_timeseries_dataset,
    build_tft_model,
    train_tft,
)

# Filter to only columns that exist in our feature set
available_cols = set(train_df.columns)

known_reals   = [c for c in TIME_VARYING_KNOWN_REALS   if c in available_cols]
unknown_reals = [c for c in TIME_VARYING_UNKNOWN_REALS if c in available_cols]
static_cats   = [c for c in STATIC_CATEGORICALS         if c in available_cols]

print(f'Known reals ({len(known_reals)}):   {known_reals}')
print(f'Unknown reals ({len(unknown_reals)}): {unknown_reals}')
print(f'Static cats ({len(static_cats)}):   {static_cats}')

Known reals (14):   ['time_idx', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'sell_price', 'is_snap', 'is_event', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'price_vs_category_mean', 'is_price_reduced']
Unknown reals (8): ['sales', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28']
Static cats (4):   ['store_id', 'dept_id', 'cat_id', 'state_id']


In [5]:
# Concatenate train+val for building the dataset (TFT sees both during dataset construction)
trainval_df = pd.concat([train_df, val_df], ignore_index=True).sort_values(['id', 'time_idx'])

training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx              = 'time_idx',
    target                = 'sales',
    group_ids             = ['id'],
    min_encoder_length    = 28,
    max_encoder_length    = 56,
    min_prediction_length = 28,
    max_prediction_length = 28,
    static_categoricals   = static_cats,
    time_varying_known_reals   = known_reals,
    time_varying_unknown_reals = unknown_reals,
    target_normalizer = GroupNormalizer(groups=['id'], transformation='softplus'),
    add_relative_time_idx = True,
    add_target_scales     = True,
)
print(f'Training dataset: {len(training_dataset):,} samples')

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, trainval_df, predict=True, stop_randomization=True
)
print(f'Validation dataset: {len(validation_dataset):,} samples')

Training dataset: 9,150,000 samples
Validation dataset: 5,000 samples


In [6]:
BATCH_SIZE = 128

train_loader = training_dataset.to_dataloader(
    train=True, batch_size=BATCH_SIZE, num_workers=2
)
val_loader = validation_dataset.to_dataloader(
    train=False, batch_size=BATCH_SIZE, num_workers=2
)

# Verify batch shapes
batch = next(iter(train_loader))
print('Batch x keys:', list(batch[0].keys()))
print('Target shape:', batch[1][0].shape)

Batch x keys: ['encoder_cat', 'encoder_cont', 'encoder_target', 'encoder_lengths', 'decoder_cat', 'decoder_cont', 'decoder_target', 'decoder_lengths', 'decoder_time_idx', 'groups', 'target_scale']
Target shape: torch.Size([128, 28])


## 9.3 Initial TFT Training (baseline configuration)

Hyperparameters from `configs/tft_config.yaml`:
- `hidden_size=64`, `attention_head_size=4`, `dropout=0.1`
- `hidden_continuous_size=16`, `output_size=7` (7 quantiles)
- `gradient_clip_val=0.1` (critical for TFT stability)

In [7]:
tft_model = build_tft_model(
    training_dataset,
    hidden_size          = 64,
    attention_head_size  = 4,
    dropout              = 0.1,
    hidden_continuous_size = 16,
    learning_rate        = 3e-3,
    log_interval         = 10,
)
print(tft_model.hparams)

"attention_head_size":               4
"categorical_groups":                {}
"causal_attention":                  True
"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'sales', 'group_ids': ['id'], 'weight': None, 'max_encoder_length': 56, 'min_encoder_length': 28, 'min_prediction_idx': np.int64(28), 'min_prediction_length': 28, 'max_prediction_length': 28, 'static_categoricals': ['store_id', 'dept_id', 'cat_id', 'state_id'], 'static_reals': None, 'time_varying_known_categoricals': None, 'time_varying_known_reals': ['time_idx', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'sell_price', 'is_snap', 'is_event', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'price_vs_category_mean', 'is_price_reduced'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['sales', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missi

In [8]:
checkpoint_cb = pl.callbacks.ModelCheckpoint(
    dirpath     = str(MODELS_DIR),
    filename    = 'tft_best',
    monitor     = 'val_loss',
    save_top_k  = 1,
    mode        = 'min',
)
early_stop_cb = pl.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 10,
    mode    = 'min',
)
# LearningRateMonitor removed: requires an active logger (incompatible with logger=False)

trainer = pl.Trainer(
    max_epochs          = 30,
    gradient_clip_val   = 0.1,
    callbacks           = [checkpoint_cb, early_stop_cb],
    enable_progress_bar = True,
    logger              = False,  # set to MLflow logger in production
    limit_train_batches = 500,    # cap batches/epoch for speed (~64K samples)
)

trainer.fit(
    tft_model,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader,
)
print(f'Best val loss: {checkpoint_cb.best_model_score:.4f}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    113 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    832 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 10.6 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 82.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 51.5 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 351 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 351 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 781                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


Best val loss: 0.5422


## 9.4 Generate TFT Predictions (Initial Model)

In [9]:
# Load best checkpoint
best_tft = TemporalFusionTransformer.load_from_checkpoint(
    str(MODELS_DIR / 'tft_best.ckpt')
)
best_tft.eval()
print('Best TFT checkpoint loaded.')

Best TFT checkpoint loaded.


In [10]:
# Quantile predictions on validation set
val_raw_preds = best_tft.predict(val_loader, mode='quantiles', return_index=True)
# val_raw_preds is a namedtuple with .output (predictions) and .index (DataFrame)
val_quantiles = val_raw_preds.output    # (N, pred_len, 7)
val_index     = val_raw_preds.index     # DataFrame with id, time_idx

print(f'Val quantile shape: {val_quantiles.shape}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Val quantile shape: torch.Size([5000, 28, 7])


In [11]:
QUANTILE_NAMES = ['p02', 'p10', 'p25', 'p50', 'p75', 'p90', 'p98']

def build_pred_df(quantiles, index_df, actual_loader, quantile_names):
    """Assemble a tidy prediction DataFrame from TFT quantile output."""
    records = []
    actuals_list = []
    for x, (y, _) in actual_loader:
        actuals_list.append(y)
    actuals = torch.cat(actuals_list, dim=0).numpy()  # (N, pred_len)

    q_np = quantiles.numpy() if isinstance(quantiles, torch.Tensor) else np.array(quantiles)
    for i, row in index_df.iterrows():
        for t in range(q_np.shape[1]):
            rec = {'id': row['id'], 'step': t}
            for j, qname in enumerate(quantile_names):
                rec[qname] = float(np.clip(q_np[i, t, j], 0, None))
            if i < len(actuals):
                rec['actual'] = float(actuals[i, t])
            records.append(rec)
    return pd.DataFrame(records)

val_pred_df = build_pred_df(val_quantiles, val_index, val_loader, QUANTILE_NAMES)
val_pred_df.to_csv(PREDS_DIR / 'tft_val_preds.csv', index=False)
print(f'TFT val predictions saved: {len(val_pred_df):,} rows')
val_pred_df.head()

TFT val predictions saved: 140,000 rows


,id,step,p02,p10,p25,p50,p75,p90,p98,actual
0,FOODS_1_017_CA_1_evaluation,0,0.0,0.000013,0.001344,0.045464,1.593378,2.681581,4.544127,0.0
1,FOODS_1_017_CA_1_evaluation,1,0.0,0.000017,0.001152,0.039134,1.559091,2.450759,4.231020,1.0
2,FOODS_1_017_CA_1_evaluation,2,0.0,0.000011,0.000616,0.023460,1.348292,2.263451,4.010246,0.0
3,FOODS_1_017_CA_1_evaluation,3,0.0,0.000011,0.000565,0.021915,1.334345,2.287582,4.069651,0.0
4,FOODS_1_017_CA_1_evaluation,4,0.0,0.000032,0.001302,0.043845,1.786437,2.704811,4.647738,1.0


In [12]:
# Build test dataset
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True).sort_values(['id','time_idx'])
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, full_df, predict=True, stop_randomization=True
)
test_loader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=2)

test_raw = best_tft.predict(test_loader, mode='quantiles', return_index=True)
test_pred_df = build_pred_df(test_raw.output, test_raw.index, test_loader, QUANTILE_NAMES)
test_pred_df.to_csv(PREDS_DIR / 'tft_test_preds.csv', index=False)
print(f'TFT test predictions saved: {len(test_pred_df):,} rows')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


TFT test predictions saved: 140,000 rows


## 10.1–10.4 Optuna Hyperparameter Tuning

Search space follows the plan:
- `hidden_size`: {32, 64, 128, 256}
- `attention_head_size`: {2, 4, 8}
- `dropout`: uniform [0.05, 0.4]
- `hidden_continuous_size`: {8, 16, 32}
- `learning_rate`: log-uniform [1e-4, 1e-2]
- `max_encoder_length`: {56, 84, 112, 168}
- `batch_size`: {32, 64, 128}

Each trial trains for ≤10 epochs; `MedianPruner` terminates poor trials early.

In [13]:
N_TRIALS     = 10
TRIAL_EPOCHS = 5
PRUNE_WARMUP = 5

# Self-contained pruning callback — replaces optuna.integration.PyTorchLightningPruningCallback
# which requires the separate `optuna-integration` package (not installed).
class _OptunaPruningCallback(pl.callbacks.Callback):
    def __init__(self, trial: optuna.Trial, monitor: str):
        self._trial   = trial
        self._monitor = monitor

    def on_validation_epoch_end(self, trainer, pl_module):
        value = trainer.callback_metrics.get(self._monitor)
        if value is None:
            return
        self._trial.report(float(value), step=trainer.current_epoch)
        if self._trial.should_prune():
            raise optuna.exceptions.TrialPruned()

def objective(trial: optuna.Trial) -> float:
    hidden_size          = trial.suggest_categorical('hidden_size', [32, 64, 128])
    attention_head_size  = trial.suggest_categorical('attention_head_size', [2, 4, 8])
    dropout              = trial.suggest_float('dropout', 0.05, 0.4)
    hidden_continuous_size = trial.suggest_categorical('hidden_continuous_size', [8, 16, 32])
    lr                   = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    max_encoder_length   = trial.suggest_categorical('max_encoder_length', [28, 56])
    batch_size           = 128  # fixed for speed

    # Rebuild dataset with this encoder length
    tr_ds = TimeSeriesDataSet(
        train_df,
        time_idx='time_idx', target='sales', group_ids=['id'],
        min_encoder_length=max_encoder_length // 2,
        max_encoder_length=max_encoder_length,
        min_prediction_length=28, max_prediction_length=28,
        static_categoricals=static_cats,
        time_varying_known_reals=known_reals,
        time_varying_unknown_reals=unknown_reals,
        target_normalizer=GroupNormalizer(groups=['id'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True,
    )
    vl_ds = TimeSeriesDataSet.from_dataset(tr_ds, trainval_df, predict=True, stop_randomization=True)

    t_loader = tr_ds.to_dataloader(train=True,  batch_size=batch_size, num_workers=2)
    v_loader = vl_ds.to_dataloader(train=False, batch_size=batch_size, num_workers=2)

    model = build_tft_model(
        tr_ds,
        hidden_size=hidden_size,
        attention_head_size=attention_head_size,
        dropout=dropout,
        hidden_continuous_size=hidden_continuous_size,
        learning_rate=lr,
    )

    pruning_cb = _OptunaPruningCallback(trial, monitor='val_loss')
    es_cb = pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min')

    trial_trainer = pl.Trainer(
        max_epochs=TRIAL_EPOCHS,
        gradient_clip_val=0.1,
        callbacks=[pruning_cb, es_cb],
        enable_progress_bar=False,
        logger=False,
        limit_train_batches=500,
    )
    try:
        trial_trainer.fit(model, train_dataloaders=t_loader, val_dataloaders=v_loader)
    except optuna.exceptions.TrialPruned:
        raise

    return float(trial_trainer.callback_metrics.get('val_loss', torch.tensor(float('inf'))))


study = optuna.create_study(
    direction  = 'minimize',
    pruner     = MedianPruner(n_warmup_steps=PRUNE_WARMUP),
    study_name = 'tft_hparam_search',
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest trial: {study.best_trial.number}')
print(f'Best val loss: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

[I 2026-04-28 10:16:12,603] A new study created in memory with name: tft_hparam_search


  0%|          | 0/10 [00:00<?, ?it/s]

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    113 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    832 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 10.6 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 82.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 51.5 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  9.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 350 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 350 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 789                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    113 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.7 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 36.3 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  280 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  178 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 66.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 66.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 66.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 66.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  132 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  132 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │ 33.0 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    256 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 82.7 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 49.5 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │ 33.3 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 66.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │ 33.3 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    903 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 777                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=5` reached.


[I 2026-04-28 13:52:05,701] Trial 9 finished with value: 0.566657304763794 and parameters: {'hidden_size': 128, 'attention_head_size': 2, 'dropout': 0.3115809894130278, 'hidden_continuous_size': 32, 'learning_rate': 0.003039378736285331, 'max_encoder_length': 28}. Best is trial 2 with value: 0.5593562722206116.

Best trial: 2
Best val loss: 0.5594
Best params: {'hidden_size': 32, 'attention_head_size': 8, 'dropout': 0.13858348921881514, 'hidden_continuous_size': 32, 'learning_rate': 0.0005628187441522603, 'max_encoder_length': 56}


In [14]:
# Plot Optuna results
try:
    fig_hist = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'optuna_history.png', dpi=150)
    plt.show()

    fig_imp = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'optuna_param_importance.png', dpi=150)
    plt.show()
except Exception as e:
    print(f'Optuna plot error (non-fatal): {e}')

## 10.4 Retrain with Best Hyperparameters (80 epochs)

Using the winning hyperparameters from the Optuna study, we retrain for up to 80 epochs
with patience=15 and save the final best checkpoint.

In [15]:
bp = study.best_params

# Rebuild dataset with optimal encoder length
best_enc_len = bp.get('max_encoder_length', 112)
final_tr_ds = TimeSeriesDataSet(
    train_df,
    time_idx='time_idx', target='sales', group_ids=['id'],
    min_encoder_length=best_enc_len // 2,
    max_encoder_length=best_enc_len,
    min_prediction_length=28, max_prediction_length=28,
    static_categoricals=static_cats,
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=unknown_reals,
    target_normalizer=GroupNormalizer(groups=['id'], transformation='softplus'),
    add_relative_time_idx=True, add_target_scales=True,
)
final_vl_ds = TimeSeriesDataSet.from_dataset(
    final_tr_ds, trainval_df, predict=True, stop_randomization=True
)
final_batch = bp.get('batch_size', 64)
final_t_loader = final_tr_ds.to_dataloader(train=True,  batch_size=final_batch, num_workers=2)
final_v_loader = final_vl_ds.to_dataloader(train=False, batch_size=final_batch, num_workers=2)

final_tft = build_tft_model(
    final_tr_ds,
    hidden_size            = bp.get('hidden_size', 64),
    attention_head_size    = bp.get('attention_head_size', 4),
    dropout                = bp.get('dropout', 0.1),
    hidden_continuous_size = bp.get('hidden_continuous_size', 16),
    learning_rate          = bp.get('learning_rate', 3e-3),
)

final_ckpt_cb = pl.callbacks.ModelCheckpoint(
    dirpath='../outputs/models', filename='tft_final_best',
    monitor='val_loss', save_top_k=1, mode='min'
)
final_es_cb = pl.callbacks.EarlyStopping(monitor='val_loss', patience=15, mode='min')

final_trainer = pl.Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[final_ckpt_cb, final_es_cb],
    enable_progress_bar=True,
    logger=False,
    limit_train_batches=500,
)
final_trainer.fit(
    final_tft,
    train_dataloaders=final_t_loader,
    val_dataloaders=final_v_loader,
)
print(f'Final best val loss: {final_ckpt_cb.best_model_score:.4f}')

Epoch 20/29 ━━                                 33/500 0:00:12 • 0:01:25 5.53it/s train_loss_step: 0.452 val_loss:  
                                                                                 0.555 train_loss_epoch: 0.464     

`Trainer.fit` stopped: `max_epochs=30` reached.


Final best val loss: 0.5472


In [16]:
# Persist best hyperparameters to tft_config.yaml
with open('../configs/tft_config.yaml', 'r') as f:
    cfg_data = yaml.safe_load(f)

cfg_data['model']['hidden_size']           = int(bp.get('hidden_size', 64))
cfg_data['model']['attention_head_size']   = int(bp.get('attention_head_size', 4))
cfg_data['model']['dropout']               = float(bp.get('dropout', 0.1))
cfg_data['model']['hidden_continuous_size']= int(bp.get('hidden_continuous_size', 16))
cfg_data['model']['learning_rate']         = float(bp.get('learning_rate', 3e-3))
cfg_data['dataset']['max_encoder_length']  = int(bp.get('max_encoder_length', 112))
cfg_data['training']['batch_size']         = int(bp.get('batch_size', 64))
cfg_data['optuna']['best_val_loss']        = float(study.best_value)

with open('../configs/tft_config.yaml', 'w') as f:
    yaml.dump(cfg_data, f, default_flow_style=False)
print('configs/tft_config.yaml updated with best Optuna hyperparameters.')

configs/tft_config.yaml updated with best Optuna hyperparameters.


## 12.1 Interpretability — Variable Importance (VSN Weights)

In [17]:
# Load final best model for interpretability
final_model = TemporalFusionTransformer.load_from_checkpoint(
    '../outputs/models/tft_final_best.ckpt'
)
final_model.eval()

# Run predictions with interpretation output
raw_preds = final_model.predict(
    final_v_loader, mode='raw', return_x=True
)
interp = final_model.interpret_output(raw_preds.output, reduction='mean')
print('Interpretation keys:', list(interp.keys()))

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Interpretation keys: ['attention', 'static_variables', 'encoder_variables', 'decoder_variables', 'encoder_length_histogram', 'decoder_length_histogram']


In [18]:
def plot_importance(importance_dict, title, fname):
    fig, ax = plt.subplots(figsize=(8, max(4, len(importance_dict) * 0.4)))
    names = list(importance_dict.keys())
    vals  = [float(v) for v in importance_dict.values()]
    sorted_pairs = sorted(zip(vals, names))
    vals_s, names_s = zip(*sorted_pairs)
    ax.barh(names_s, vals_s, color='steelblue')
    ax.set_xlabel('Importance Weight')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / fname, dpi=150)
    plt.show()
    print(f'Saved {fname}')


# Static variable importance
if 'static_variables' in interp:
    plot_importance(
        dict(zip(static_cats, interp['static_variables'].cpu().numpy())),
        'Static Variable Importance (VSN)',
        'tft_static_importance.png',
    )

# Encoder variable importance
if 'encoder_variables' in interp:
    enc_vars = known_reals + unknown_reals
    enc_imp  = interp['encoder_variables'].cpu().numpy()
    plot_importance(
        dict(zip(enc_vars[:len(enc_imp)], enc_imp)),
        'Encoder Variable Importance (VSN)',
        'tft_encoder_importance.png',
    )

# Decoder variable importance
if 'decoder_variables' in interp:
    dec_imp = interp['decoder_variables'].cpu().numpy()
    plot_importance(
        dict(zip(known_reals[:len(dec_imp)], dec_imp)),
        'Decoder Variable Importance (VSN)',
        'tft_decoder_importance.png',
    )

Saved tft_static_importance.png
Saved tft_encoder_importance.png
Saved tft_decoder_importance.png


## 12.2 Attention Heatmaps

In [19]:
if 'attention' in interp:
    attn = interp['attention'].cpu().numpy()
    fig, ax = plt.subplots(figsize=(12, 4))
    if attn.ndim == 1:
        # 1-D: attention averaged over decoder steps, heads, and series
        ax.bar(range(len(attn)), attn, color='steelblue', alpha=0.8)
        ax.set_xlabel('Encoder timestep (past, relative)')
        ax.set_ylabel('Attention weight')
        ax.set_title('TFT Attention Weights (averaged over decoder steps, heads, series)')
    else:
        # 2-D: (encoder_len, decoder_len) — full heatmap
        im = ax.imshow(attn.T, aspect='auto', origin='lower', cmap='YlOrRd')
        ax.set_xlabel('Encoder timestep (past)')
        ax.set_ylabel('Forecast horizon step')
        ax.set_title('TFT Attention Heatmap (averaged over heads and series)')
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'tft_attention_heatmap.png', dpi=150)
    plt.show()
    print(f'Saved tft_attention_heatmap.png  (attn shape={attn.shape})')
else:
    print('Attention weights not available in this pytorch-forecasting version — skipping.')

Saved tft_attention_heatmap.png  (attn shape=(28,))


## 12.3 Prediction Interval Calibration

In [20]:
# Reload final predictions with all quantiles
final_val_raw = final_model.predict(final_v_loader, mode='quantiles', return_index=True)
q_preds = final_val_raw.output.cpu().numpy()  # (N, 28, 7)

# Gather actuals
all_actuals = []
for x, (y, _) in final_v_loader:
    all_actuals.append(y.cpu().numpy())
all_actuals = np.concatenate(all_actuals, axis=0)  # (N, 28)

print(f'Quantile predictions shape: {q_preds.shape}')
print(f'Actuals shape: {all_actuals.shape}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Quantile predictions shape: (5000, 28, 7)
Actuals shape: (5000, 28)


In [21]:
# Empirical coverage of the P10-P90 band (target: ~80%)
if q_preds.shape[0] == all_actuals.shape[0]:
    p10 = np.clip(q_preds[:, :, 1], 0, None)  # index 1 = P10
    p90 = np.clip(q_preds[:, :, 5], 0, None)  # index 5 = P90
    inside = ((all_actuals >= p10) & (all_actuals <= p90)).mean()
    print(f'Empirical coverage P10-P90: {inside:.2%}  (target ~80%)')

    # Calibration curve
    quantile_levels = [0.02, 0.10, 0.25, 0.50, 0.75, 0.90, 0.98]
    empirical_fracs = []
    for qi, q_level in enumerate(quantile_levels):
        q_pred = np.clip(q_preds[:, :, qi], 0, None)
        frac   = float((all_actuals <= q_pred).mean())
        empirical_fracs.append(frac)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.plot(quantile_levels, empirical_fracs, 'o-', color='steelblue', label='TFT')
    ax.set_xlabel('Nominal quantile level')
    ax.set_ylabel('Empirical fraction below quantile')
    ax.set_title('TFT Quantile Calibration Curve')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'tft_calibration_curve.png', dpi=150)
    plt.show()
    print('Saved tft_calibration_curve.png')
else:
    print('Shape mismatch between predictions and actuals — skipping calibration.')

Empirical coverage P10-P90: 29.83%  (target ~80%)
Saved tft_calibration_curve.png


In [22]:
# Plot 6 representative series: actual vs P50 with P10-P90 band
if q_preds.shape[0] >= 6:
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    for ax, sample_idx in zip(axes.flat, range(6)):
        actual = all_actuals[sample_idx]
        p10_s  = np.clip(q_preds[sample_idx, :, 1], 0, None)
        p50_s  = np.clip(q_preds[sample_idx, :, 3], 0, None)
        p90_s  = np.clip(q_preds[sample_idx, :, 5], 0, None)
        steps  = np.arange(1, 29)
        ax.plot(steps, actual,  label='Actual',  color='black')
        ax.plot(steps, p50_s,   label='P50 Forecast', color='steelblue')
        ax.fill_between(steps, p10_s, p90_s, alpha=0.25,
                        color='steelblue', label='P10-P90 Band')
        ax.set_title(f'Sample {sample_idx}')
        ax.legend(fontsize=7)
    plt.suptitle('TFT 28-Day Forecasts with Prediction Intervals')
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'tft_forecast_samples.png', dpi=150)
    plt.show()
    print('Saved tft_forecast_samples.png')

Saved tft_forecast_samples.png
